# Analysis code for HREX simulations

## Compute the FES in the polar system from unscaled replica 

In [3]:
import MDAnalysis as mda
import numpy as np
from MDAnalysis.analysis import align
import matplotlib.pyplot as plt
import pandas as pd 
import seaborn as sns

In [4]:
u = mda.Universe('ref.pdb', 'meta_cluter.xtc')
reference = mda.Universe('../ref_bb.pdb')
reference_frame = reference.select_atoms('backbone').positions.copy()
reference_all=reference.select_atoms('all')

In [5]:
align.AlignTraj(u, reference, select='backbone', in_memory=True).run()

In [6]:
def get_normal_vector(coords):
    geometric_center = np.mean(coords, axis=0)
    traslated_coords = coords - geometric_center
    N = len(coords)
    R_prime = np.zeros(3)
    R_double_prime = np.zeros(3)
    for j in range(N):
        R_prime += traslated_coords[j] * np.cos(2 * np.pi * j / N)
        R_double_prime += traslated_coords[j] * np.sin(2 * np.pi * j / N)
    normal_vector = np.cross(R_prime, R_double_prime)
    normal_vector /= np.linalg.norm(normal_vector)  
    return normal_vector

In [7]:
normal_vectors=[]
for ts in u.trajectory:
    ring=u.select_atoms('id 1978 1968 1971 1973 1975 1977')
    coords=ring.positions
    normal_vectors.append(get_normal_vector(coords))

In [8]:
np.savetxt('normal_vectors.txt', normal_vectors, header='Nx Ny Nz', comments='#')

In [9]:
normal_vectors=np.loadtxt('normal_vectors.txt')
normal_vectors

array([[-0.45229073, -0.22229027,  0.86372457],
       [-0.3709867 , -0.35446132,  0.85832747],
       [ 0.32667181, -0.92801292, -0.17910207],
       ...,
       [-0.59701118, -0.67724759, -0.43001553],
       [ 0.16869738,  0.43777959, -0.88311394],
       [-0.73983656,  0.26016494, -0.62044828]])

In [10]:
ca_atoms=reference.select_atoms('name CA')

In [11]:
projections=[]
for i in range(len(normal_vectors)):
    projections.append(np.dot(ca_atoms.principal_axes(), normal_vectors[i]))
projections=np.array(projections)

In [12]:
def cartesian_to_spherical(cartesian_coords):
    x, y, z = cartesian_coords
    r = np.linalg.norm(cartesian_coords)  
    phi = np.degrees(np.arctan2(y, x))  
    theta = np.degrees(np.arccos(z / r)) 
    return (phi, theta, r)

In [13]:
polar=[]
for i in projections:
    polar.append(cartesian_to_spherical(i))

polar=np.array(polar).transpose()
phi, theta, r = polar

In [14]:
cv = np.loadtxt('DIST')[:,1]

In [15]:
polar_system = np.loadtxt('polar_data.txt')

In [16]:
df = pd.DataFrame(polar_system, columns=['CV', 'phi', 'theta'])

In [17]:
df.to_csv('fes_polar_system.csv', index=False, encoding='utf-8')

## Compute the FES from all replicas in 3D space 